# Greenwashing Detector — Walkthrough

End-to-end demo of the multi-agent pipeline:

1. **Detector** — finds suspicious phrases
2. **Classifier** — decides keep / replace / delete
3. **Rewriter** — generates honest replacements and assembles the final document

Every step is auto-traced to LangSmith if `LANGSMITH_API_KEY` is set in `.env`.

## 1. Setup
Make sure Ollama is running and the model is pulled:
```bash
ollama pull llama3.2:3b
ollama serve   # in a separate terminal
```

In [ ]:
import sys
sys.path.insert(0, "../src")  # so we can import without `pip install -e .`

from greenwashing.graph import analyse

## 2. Run on a single document

In [ ]:
doc = (
    "Our new packaging is 100% eco-friendly and helps save the planet "
    "for future generations. We use 30% recycled PET and partner with "
    "RSPO-certified suppliers."
)

result = analyse(doc)

print("ORIGINAL :", result.original_text)
print("REWRITTEN:", result.final_text)
print()
for d in result.decisions:
    print(f"  - [{d.action.value}] {d.phrase!r} — {d.justification}")

## 3. Inspect intermediate state
Check the LangSmith UI for the full trace tree (detector → classifier → rewriter, with per-node latency, tokens, and inputs/outputs).

In [ ]:
for s in result.spans:
    print(f"{s.confidence:.2f}  {s.phrase!r}\n      └─ {s.reason}")

## 4. Run the evaluation suite
Seeds the dataset on the first run, then evaluates with custom + built-in + LLM-judge evaluators.

In [ ]:
sys.path.insert(0, "..")  # so `evals` is importable
from evals.run_eval import main as run_eval

run_eval(dataset_name="greenwashing-eval", seed=True)

## 5. Failure handling demo
Force a parse failure by feeding the detector something unusual; the graph still returns a valid `AnalysisResult` with `errors` populated.

In [ ]:
edge = analyse("")  # empty doc
print("errors:", edge.errors)
print("final :", repr(edge.final_text))